# **TP Intelligence Artificielle - SAT et CSP**

# **Partie 0 : Visualisation**

Voici une fonction pour visualiser les sudokus.

In [1]:
from IPython.display import HTML

def visualize_sudoku(grid):
    html = "<table>"
    for i in range(9):
        html += "<tr>"
        for j in range(9):
            value = grid[i][j]
            cell_style = "width: 30px; height: 30px; text-align: center; font-size: 20px; border: 1px solid black;"
            if i % 3 == 0 and i != 0:  # Horizontal lines
                cell_style += "border-top: 5px solid black;"
            if j % 3 == 0 and j != 0:  # Vertical lines
                cell_style += "border-left: 5px solid black;"

            if value == 0:
                html += f"<td style='background-color: white; {cell_style}'> </td>"  # Case vide
            else:
                html += f"<td style='background-color: rgb(226, 114, 91); {cell_style}'>{value}</td>"
        html += "</tr>"
    html += "</table>"
    display(HTML(html))

### **Exercice 1**
Écrivez un Sudoku sous forme d’une liste de listes (9 listes de 9 élements chacune où la valeur 0 represente une caise vide), puis utilisez la fonction visualize_sudoku pour l’afficher.

In [2]:

# Exemple d'utilisation
sudoku_grid = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]

visualize_sudoku(sudoku_grid)


5,3,,,7,,,,
6,,,1,9,5,,,
,9,8,,,,,6,
8,,,,6,,,,3
4,,,8,,3,,,1
7,,,,2,,,,6
,6,,,,,2,8,
,,,4,1,9,,,5
,,,,8,,,7,9


# **Partie 1 : Modélisation**

Nous allons modéliser le Sudoku comme un problème SAT et CSP, puis utiliser les méthodes propres à chaque modélisation pour la résolution.
Pour cela, nous utiliserons deux bibliothèques : **python-sat** et **python-constraint**.

Exécutez le bloc de code suivant pour les installer.
Pour les installer sur vos PC, il suffit d’exécuter les mêmes lignes (sans le '!' au début) dans une console de commandes (CMD).



In [3]:

# Cellule compatible hors-ligne.
# Si vous avez internet AVANT le controle, vous pouvez installer une fois:
# !pip install python-sat python-constraint

PYSAT_AVAILABLE = False
CONSTRAINT_AVAILABLE = False

try:
    from pysat.formula import CNF
    from pysat.solvers import Glucose3
    PYSAT_AVAILABLE = True
    print("PySAT disponible: solveur SAT actif.")
except Exception as e:
    print("PySAT indisponible -> fallback backtracking pour la partie SAT.")

try:
    from constraint import Problem, AllDifferentConstraint
    CONSTRAINT_AVAILABLE = True
    print("python-constraint disponible: solveur CSP actif.")
except Exception as e:
    print("python-constraint indisponible -> fallback backtracking pour la partie CSP.")



Hi there!  pysat will nominally store data in a 'pysatData' directory which needs to be assigned. Please run `pysat.params['data_dirs'] = path` where path specifies one or more existing top-level directories that may be used to store science data. `path` may either be a single string or a list of strings.
PySAT indisponible -> fallback backtracking pour la partie SAT.
python-constraint disponible: solveur CSP actif.


## **1.1 Modélisation SAT**

Pour modéliser le Sudoku comme un problème SAT, nous utiliserons la méthode CNF de la bibliothèque pysat.formula.
CNF permet de définir les clauses logiques représentant les contraintes du problème.
Pour les variables, nous utiliserons un dictionnaire.

In [4]:

# Fallback CNF minimal si PySAT n'est pas present
if not PYSAT_AVAILABLE:
    class CNF:
        def __init__(self):
            self.clauses = []

        def append(self, clause):
            self.clauses.append(list(clause))


class Sudoku_SAT:
    def __init__(self, grid):
        self.grid = grid
        self.cnf = CNF()
        self.var_map = {}

        # Mapping (i, j, k) -> variable SAT
        var_count = 1
        for i in range(9):
            for j in range(9):
                for k in range(1, 10):
                    self.var_map[(i, j, k)] = var_count
                    var_count += 1

        # 1) Au moins une valeur par case
        for i in range(9):
            for j in range(9):
                self.cnf.append([self.var_map[(i, j, k)] for k in range(1, 10)])

        # 2) Au plus une valeur par case
        for i in range(9):
            for j in range(9):
                for k1 in range(1, 10):
                    for k2 in range(k1 + 1, 10):
                        self.cnf.append([-self.var_map[(i, j, k1)], -self.var_map[(i, j, k2)]])

        # 3) Chaque ligne contient 1..9
        for i in range(9):
            for k in range(1, 10):
                self.cnf.append([self.var_map[(i, j, k)] for j in range(9)])

        # 4) Chaque colonne contient 1..9
        for j in range(9):
            for k in range(1, 10):
                self.cnf.append([self.var_map[(i, j, k)] for i in range(9)])

        # 5) Chaque sous-grille 3x3 contient 1..9
        for sr in range(0, 9, 3):
            for sc in range(0, 9, 3):
                for k in range(1, 10):
                    self.cnf.append([
                        self.var_map[(sr + di, sc + dj, k)]
                        for di in range(3)
                        for dj in range(3)
                    ])

        # 6) Contraintes initiales
        for i in range(9):
            for j in range(9):
                value = self.grid[i][j]
                if value != 0:
                    self.cnf.append([self.var_map[(i, j, value)]])


### **Exercice 2**
Créez une fonction qui prend une grille de Sudoku en entrée et utilise le solveur Glucose3 de la bibliothèque pysat.solver pour résoudre le jeu (pensez à bien importer Glucosa3). Votre fonction doit :

1. Créer un objet sudoku_sat pour modéliser le problème.
2. Ajouter au solveur les contraintes du Sudoku sous forme de clauses logiques.
3. Vérifier si le Sudoku est résoluble en utilisant le solveur SAT.
4. Trouver une solution si le Sudoku est résoluble.
5. Interpréter la solution retournée par le solveur pour remplir la grille :
  * La solution sera une liste de taille 9 × 9 × 10, contenant des valeurs positives et négatives.
  * Une valeur positive signifie que la variable (i, j, k) est True, ce qui indique que la case (i, j) doit contenir la valeur k.

In [5]:

def _find_empty(grid):
    for i in range(9):
        for j in range(9):
            if grid[i][j] == 0:
                return i, j
    return None


def _is_valid(grid, row, col, val):
    if any(grid[row][j] == val for j in range(9)):
        return False
    if any(grid[i][col] == val for i in range(9)):
        return False

    sr, sc = (row // 3) * 3, (col // 3) * 3
    for i in range(sr, sr + 3):
        for j in range(sc, sc + 3):
            if grid[i][j] == val:
                return False
    return True


def solve_sudoku_backtracking(grid):
    """Solveur hors-ligne de secours (sans dependance externe)."""
    grid = [row[:] for row in grid]

    def backtrack():
        empty = _find_empty(grid)
        if empty is None:
            return True

        row, col = empty
        for val in range(1, 10):
            if _is_valid(grid, row, col, val):
                grid[row][col] = val
                if backtrack():
                    return True
                grid[row][col] = 0
        return False

    return grid if backtrack() else None


def solve_sudoku_sat(grid):
    """Resolution SAT (PySAT), sinon fallback backtracking."""
    if not PYSAT_AVAILABLE:
        return solve_sudoku_backtracking(grid)

    sudoku_sat = Sudoku_SAT(grid)
    solver = Glucose3()
    for clause in sudoku_sat.cnf.clauses:
        solver.add_clause(clause)

    if not solver.solve():
        solver.delete()
        return None

    model = set(lit for lit in solver.get_model() if lit > 0)
    solved = [[0] * 9 for _ in range(9)]

    for (i, j, k), var in sudoku_sat.var_map.items():
        if var in model:
            solved[i][j] = k

    solver.delete()
    return solved


### **Exercice 3**
Utilisez votre fonction pour resoudre le sudoku ci-dessous:

    [0, 0, 0, 0, 7, 0, 0, 0, 0]
    [6, 0, 0, 1, 9, 5, 0, 0, 0]
    [0, 9, 8, 0, 0, 0, 0, 6, 0]
    [8, 0, 0, 0, 6, 0, 0, 0, 3]
    [4, 0, 0, 8, 0, 3, 0, 0, 1]
    [7, 0, 0, 0, 2, 0, 0, 0, 6]
    [0, 6, 0, 0, 0, 0, 2, 8, 0]
    [0, 0, 0, 4, 1, 9, 0, 0, 5]
    [0, 0, 0, 0, 8, 0, 0, 7, 9]

In [6]:

# Exercice 3 - Test solveur SAT
sudoku_ex3 = [
    [0, 0, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]

solution_sat = solve_sudoku_sat(sudoku_ex3)

print("Grille initiale:")
visualize_sudoku(sudoku_ex3)

if solution_sat is None:
    print("Aucune solution trouvée.")
else:
    print("Solution SAT:")
    visualize_sudoku(solution_sat)


Grille initiale:


,,,,7,,,,
6,,,1,9,5,,,
,9,8,,,,,6,
8,,,,6,,,,3
4,,,8,,3,,,1
7,,,,2,,,,6
,6,,,,,2,8,
,,,4,1,9,,,5
,,,,8,,,7,9


Solution SAT:


3,4,5,6,7,8,9,1,2
6,7,2,1,9,5,3,4,8
1,9,8,3,4,2,5,6,7
8,5,9,7,6,1,4,2,3
4,2,6,8,5,3,7,9,1
7,1,3,9,2,4,8,5,6
9,6,1,5,3,7,2,8,4
2,8,7,4,1,9,6,3,5
5,3,4,2,8,6,1,7,9


## **1.2 Modélisation CSP**

### **Exercice 4**

Créez une fonction solve_sudoku_csp qui prend une grille de Sudoku en entrée, la modélise comme un problème CSP et la résout. Votre fonction doit :
1. Créer une instance du problème CSP
  * Utilisez la classe Problem de la bibliothèque python-constraint pour représenter le Sudoku.
2. Définir les variables du problème
  * Utilisez la méthode addVariable pour ajouter les variables au problème, sachant que :
    * Chaque case de la grille est une variable identifiée par un tuple (ligne, colonne).
    * Le domaine de chaque variable correspond aux valeurs possibles (de 1 à 9).
    * Si une case contient déjà un chiffre (non nul), son domaine est restreint à cette valeur unique.
3. Définir les contraintes du Sudoku (les valeurs doivent être uniques dans chaque ligne, colonne et région 3×3). Pour cela, utilisez la méthode addConstraint avec AllDifferentConstraint() pour assurer ces contraintes.
4. Résoudre le problème. Pour cela utilisez la méthode getSolutions() pour obtenir toutes les solutions possibles.
5. Gérer les solutions
  * Si des solutions existent, les stocker dans une liste.
  * Sinon, indiquer qu’aucune solution n’a été trouvée.

In [7]:

def solve_sudoku_csp(grid):
    """Resolution CSP avec python-constraint, sinon fallback backtracking."""
    if not CONSTRAINT_AVAILABLE:
        return solve_sudoku_backtracking(grid)

    problem = Problem()

    # Variables et domaines
    for i in range(9):
        for j in range(9):
            if grid[i][j] == 0:
                problem.addVariable((i, j), range(1, 10))
            else:
                problem.addVariable((i, j), [grid[i][j]])

    # Contraintes de lignes
    for i in range(9):
        problem.addConstraint(AllDifferentConstraint(), [(i, j) for j in range(9)])

    # Contraintes de colonnes
    for j in range(9):
        problem.addConstraint(AllDifferentConstraint(), [(i, j) for i in range(9)])

    # Contraintes de sous-grilles 3x3
    for sr in range(0, 9, 3):
        for sc in range(0, 9, 3):
            block = [(sr + di, sc + dj) for di in range(3) for dj in range(3)]
            problem.addConstraint(AllDifferentConstraint(), block)

    solution = problem.getSolution()
    if solution is None:
        return None

    solved = [[0] * 9 for _ in range(9)]
    for i in range(9):
        for j in range(9):
            solved[i][j] = solution[(i, j)]

    return solved


### **Exercice 5**
Utilisez votre fonction pour resoudre le jeu suivant.

In [8]:
# Grille de Sudoku (0 représente une case vide)
grid = [[5, 3, 0, 0, 7, 0, 0, 0, 0],
        [6, 0, 0, 1, 9, 5, 0, 0, 0],
        [0, 9, 8, 0, 0, 0, 0, 6, 0],
        [8, 0, 0, 0, 6, 0, 0, 0, 3],
        [4, 0, 0, 8, 0, 3, 0, 0, 1],
        [7, 0, 0, 0, 2, 0, 0, 0, 6],
        [0, 6, 0, 0, 0, 0, 2, 8, 0],
        [0, 0, 0, 4, 1, 9, 0, 0, 5],
        [0, 0, 0, 0, 8, 0, 0, 7, 9]]

In [9]:

# Exercice 5 - Test solveur CSP
solution_csp = solve_sudoku_csp(grid)

print("Grille initiale:")
visualize_sudoku(grid)

if solution_csp is None:
    print("Aucune solution trouvée.")
else:
    print("Solution CSP:")
    visualize_sudoku(solution_csp)


Grille initiale:


5,3,,,7,,,,
6,,,1,9,5,,,
,9,8,,,,,6,
8,,,,6,,,,3
4,,,8,,3,,,1
7,,,,2,,,,6
,6,,,,,2,8,
,,,4,1,9,,,5
,,,,8,,,7,9


Solution CSP:


5,3,4,6,7,8,9,1,2
6,7,2,1,9,5,3,4,8
1,9,8,3,4,2,5,6,7
8,5,9,7,6,1,4,2,3
4,2,6,8,5,3,7,9,1
7,1,3,9,2,4,8,5,6
9,6,1,5,3,7,2,8,4
2,8,7,4,1,9,6,3,5
3,4,5,2,8,6,1,7,9


# **Partie 2 : Comparaison CSP vs SAT**

Nous allons comparer les deux solveurs sur 10 sudokus differents. Pour cela, voici 10 grilles.

In [10]:
grid1 = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
]

In [11]:
grid2 = [
    [0, 0, 3, 0, 2, 0, 6, 0, 0],
    [9, 0, 0, 3, 0, 5, 0, 0, 1],
    [0, 0, 1, 8, 0, 6, 4, 0, 0],
    [0, 0, 8, 1, 0, 2, 9, 0, 0],
    [7, 0, 0, 0, 0, 0, 0, 0, 8],
    [0, 0, 6, 7, 0, 8, 2, 0, 0],
    [0, 0, 2, 6, 0, 9, 5, 0, 0],
    [8, 0, 0, 2, 0, 3, 0, 0, 9],
    [0, 0, 5, 0, 1, 0, 3, 0, 0]
]

In [12]:
grid3 = [
    [0, 2, 0, 6, 0, 8, 0, 0, 0],
    [5, 8, 0, 0, 0, 9, 7, 0, 0],
    [0, 0, 0, 0, 4, 0, 0, 0, 0],
    [3, 7, 0, 0, 0, 0, 5, 0, 0],
    [6, 0, 0, 0, 0, 0, 0, 0, 4],
    [0, 0, 8, 0, 0, 0, 0, 1, 3],
    [0, 0, 0, 0, 2, 0, 0, 0, 0],
    [0, 0, 9, 8, 0, 0, 0, 3, 6],
    [0, 0, 0, 3, 0, 6, 0, 9, 0]
]

In [13]:
grid4 = [
    [0, 0, 5, 3, 0, 0, 0, 0, 0],
    [8, 0, 0, 0, 0, 0, 0, 2, 0],
    [0, 7, 0, 0, 1, 0, 5, 0, 0],
    [4, 0, 0, 0, 0, 5, 3, 0, 0],
    [0, 1, 0, 0, 7, 0, 0, 0, 6],
    [0, 0, 3, 2, 0, 0, 0, 8, 0],
    [0, 6, 0, 5, 0, 0, 0, 0, 9],
    [0, 0, 4, 0, 0, 0, 0, 3, 0],
    [0, 0, 0, 0, 0, 9, 7, 0, 0]
]

In [14]:
grid5 = [
    [8, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 3, 6, 0, 0, 0, 0, 0],
    [0, 7, 0, 0, 9, 0, 2, 0, 0],
    [0, 5, 0, 0, 0, 7, 0, 0, 0],
    [0, 0, 0, 0, 4, 5, 7, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 3, 0],
    [0, 0, 1, 0, 0, 0, 0, 6, 8],
    [0, 0, 8, 5, 0, 0, 0, 1, 0],
    [0, 9, 0, 0, 0, 0, 4, 0, 0]
]

In [15]:
grid6 = [
    [0, 0, 0, 6, 0, 0, 4, 0, 0],
    [7, 0, 0, 0, 0, 3, 6, 0, 0],
    [0, 0, 0, 0, 9, 1, 0, 8, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 5, 0, 1, 8, 0, 0, 0, 3],
    [0, 0, 0, 3, 0, 6, 0, 4, 5],
    [0, 4, 0, 2, 0, 0, 0, 6, 0],
    [9, 0, 3, 0, 0, 0, 0, 0, 0],
    [0, 2, 0, 0, 0, 0, 1, 0, 0]
]

In [16]:
grid7 = [
    [0, 2, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 6, 0, 0, 0, 0, 3],
    [0, 7, 4, 0, 8, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 3, 0, 0, 2],
    [0, 8, 0, 0, 4, 0, 0, 1, 0],
    [6, 0, 0, 5, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 7, 8, 0],
    [5, 0, 0, 0, 0, 9, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 4, 0]
]

In [17]:
grid8 = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 3, 0, 8, 5],
    [0, 0, 1, 0, 2, 0, 0, 0, 0],
    [0, 0, 0, 5, 0, 7, 0, 0, 0],
    [0, 0, 4, 0, 0, 0, 1, 0, 0],
    [0, 9, 0, 0, 0, 0, 0, 0, 0],
    [5, 0, 0, 0, 0, 0, 0, 7, 3],
    [0, 0, 2, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 4, 0, 0, 0, 9]
]

In [18]:
grid9 = [
    [0, 0, 0, 2, 6, 0, 7, 0, 1],
    [6, 8, 0, 0, 7, 0, 0, 9, 0],
    [1, 9, 0, 0, 0, 4, 5, 0, 0],
    [8, 2, 0, 1, 0, 0, 0, 4, 0],
    [0, 0, 4, 6, 0, 2, 9, 0, 0],
    [0, 5, 0, 0, 0, 3, 0, 2, 8],
    [0, 0, 9, 3, 0, 0, 0, 7, 4],
    [0, 4, 0, 0, 5, 0, 0, 3, 6],
    [7, 0, 3, 0, 1, 8, 0, 0, 0]
]

In [19]:
grid10 = [
    [0, 2, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 6, 0, 0, 0, 0, 3],
    [0, 7, 4, 0, 8, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 3, 0, 0, 2],
    [0, 8, 0, 0, 4, 0, 0, 1, 0],
    [6, 0, 0, 5, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 7, 8, 0],
    [5, 0, 0, 0, 0, 9, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 4, 0]
]

### **Exercice 6**

Créez une fonction comparer_solveurs qui prend comme entrée une liste de grilles, résout tous les jeux avec les deux solveurs, enregistre le temps d'exécution à chaque fois, et retourne les résultats sous forme de DataFrame.

Utilisez cette fonction pour résoudre tous les jeux et affichez ensuite les résultats.

In [ ]:

import time
import pandas as pd


def comparer_solveurs(grids):
    """Compare SAT et CSP sur une liste de grilles."""
    rows = []

    for idx, g in enumerate(grids, start=1):
        g_copy_sat = [row[:] for row in g]
        g_copy_csp = [row[:] for row in g]

        t0 = time.time()
        sat_solution = solve_sudoku_sat(g_copy_sat)
        sat_time = time.time() - t0

        t0 = time.time()
        csp_solution = solve_sudoku_csp(g_copy_csp)
        csp_time = time.time() - t0

        rows.append({
            'grille': idx,
            'sat_resolu': sat_solution is not None,
            'temps_sat_s': sat_time,
            'csp_resolu': csp_solution is not None,
            'temps_csp_s': csp_time,
        })

    return pd.DataFrame(rows)


grids = [grid1, grid2, grid3, grid4, grid5, grid6, grid7, grid8, grid9, grid10]
df_results = comparer_solveurs(grids)
df_results


In [ ]:
from tabulate import tabulate
print(tabulate(df_results, headers = 'keys', tablefmt = 'pretty'))


## Corrige detaille - SAT vs CSP

### Idee SAT
- On transforme le Sudoku en formule logique CNF.
- Le solveur SAT cherche une affectation vraie/fausse satisfaisant toutes les clauses.

### Idee CSP
- On modele chaque case comme une variable avec un domaine.
- Les contraintes `AllDifferent` imposent l'unicite sur lignes, colonnes et blocs.

### Pourquoi garder un fallback backtracking
En controle sans internet, il se peut que les librairies ne soient pas installees. Le fallback garantit une resolution locale minimale.



## FAQ rapide (TP3)

**Q1. SAT est-il toujours plus rapide que CSP ?**
Pas toujours. Cela depend de l'instance et du solveur.

**Q2. Pourquoi dupliquer la grille avant resolution ?**
Pour eviter les effets de bord entre deux solveurs compares sur la meme instance.

**Q3. Le backtracking est-il SAT ou CSP ?**
C'est une technique generale de recherche, utilisable dans les deux cadres.
